# Riverine flood risk by province

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RiskThinking/crc-docs/blob/main/notebooks/flood_risk_by_province.ipynb) · [GitHub preview](https://github.com/RiskThinking/crc-docs/blob/main/notebooks/flood_risk_by_province.ipynb)

**crc-sdk** turns a public flood-hazard raster and free, open geographic layers into one H3-joined risk view, in a few lines each:

1. **Hazard** — use the fluent `HazardDataset.efas(...)` workflow to resolve JRC's current European flood release, cache Rhine-corridor AOI crops, and materialize canonical H3 curves.
2. **Geography** — polyfill Germany's ADM1 (state/province) boundaries from [geoBoundaries](https://www.geoboundaries.org) (CC BY 4.0) to the same H3 resolution (`H3Indexer`).
3. **Join** — one DuckDB `JOIN` on the shared H3 cell id turns per-cell hazard depths into a per-province risk view.
4. **Places** — layer in [Overture Maps](https://overturemaps.org) (community-governed, open data) places, H3-indexed the same way, to see which real-world locations sit inside flood-exposed cells.
5. **Visualize** — Plotly maps and a ranked comparison.

Nothing is bundled with this notebook — hazard, boundary, and places data are fetched live on first run, then reused from `data/`. Only basemap tiles still require network access on warm runs.

**Workflow:** [AI skill](https://github.com/RiskThinking/crc-docs/blob/main/.agents/skills/crc-screen-mortgage-flood/SKILL.md) · [Python pipeline](https://github.com/RiskThinking/crc-docs/blob/main/pipelines/flood_admin_pipeline.py) · [Setup and tested versions](https://github.com/RiskThinking/crc-docs/blob/main/ai-playbooks/docs/setup.md) · [All problems](https://github.com/RiskThinking/crc-docs/blob/main/README.md#choose-a-problem)


## 0. Setup

In [1]:
#@title Install dependencies and prepare this notebook
import hashlib
import importlib
import importlib.metadata
import importlib.util
import os
from pathlib import Path
import subprocess
import sys
import sysconfig
import urllib.request

if sys.version_info < (3, 12):
    raise RuntimeError("Use a Python 3.12+ runtime (Colab: Runtime > Change runtime type).")

# Install into this kernel's Python, not a separate shell environment.
REQUIREMENTS = ['crc-sdk[geometry,raster]==0.7.1', 'crc-framework==0.2.5', 'duckdb>=1.4.5,<2', 'numpy>=1.26,<3', 'pandas>=2.2,<4', 'plotly==6.9.0', 'anywidget>=0.9,<1', 'ipywidgets>=8.1,<9', 'nbformat>=5.10,<6', 'tippecanoe==2.72.0']
loaded_versions = {
    module: getattr(sys.modules[module], "__version__", None)
    for module in ("numpy", "pandas", "pyarrow", "plotly", "duckdb", "crc_sdk", "crc_framework")
    if module in sys.modules
}
if importlib.util.find_spec("pip") is None:
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
    *REQUIREMENTS,
])
importlib.invalidate_caches()
for module, loaded in loaded_versions.items():
    installed = importlib.metadata.version(module.replace("_", "-"))
    if loaded is not None and loaded != installed:
        raise RuntimeError("Packages changed after import. Restart the kernel/session, then run all cells again.")
os.environ["PATH"] = sysconfig.get_path("scripts") + os.pathsep + os.environ.get("PATH", "")
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False

# A local checkout keeps its files; standalone/Colab runs download only required resources.
RESOURCE_REVISION = '52ff7a93028cc1e64f3b1df856851650df825dc8'
RESOURCE_SHA256 = {}
REPO_ROOT = next((
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "pyproject.toml").is_file()
    and 'name = "crc-docs"' in (folder / "pyproject.toml").read_text()
), None)
if REPO_ROOT is None:
    REPO_ROOT = next((
        folder for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / ".crc-notebook-revision").is_file()
        and (folder / ".crc-notebook-revision").read_text() == RESOURCE_REVISION
    ), Path.cwd() / ".crc-docs" / RESOURCE_REVISION)
    REPO_ROOT.mkdir(parents=True, exist_ok=True)
    (REPO_ROOT / ".crc-notebook-revision").write_text(RESOURCE_REVISION)
    for relative, expected_hash in RESOURCE_SHA256.items():
        target = REPO_ROOT / relative
        if not target.exists() or hashlib.sha256(target.read_bytes()).hexdigest() != expected_hash:
            url = f"https://raw.githubusercontent.com/RiskThinking/crc-docs/{RESOURCE_REVISION}/{relative}"
            with urllib.request.urlopen(url, timeout=120) as response:
                content = response.read()
            if hashlib.sha256(content).hexdigest() != expected_hash:
                raise RuntimeError(f"Resource checksum mismatch: {relative}")
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)
NOTEBOOK_DIR = REPO_ROOT / "notebooks"
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(NOTEBOOK_DIR)

print(f"Ready: Python {sys.version.split()[0]}, crc-sdk {importlib.metadata.version('crc-sdk')}")
print(f"Working directory: {NOTEBOOK_DIR}")


Ready: Python 3.12.10, crc-sdk 0.7.1
Working directory: /Users/woozyking/rtai/crc-docs/notebooks


In [2]:
import json
import urllib.request
from pathlib import Path

from IPython.display import FileLink, clear_output, display

import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import pyarrow.parquet as pq
from shapely.geometry import mapping

from crc_sdk.connectors.duckdb import DuckDBConnection
from crc_sdk.geometry import (
    AREAS,
    POINTS,
    POLYGONS,
    FormatAdapter,
    GeoFormat,
    H3Indexer,
    PolyfillMode,
    PMTilesBuild,
    cell_polygon,
)
from crc_sdk.workflows import (
    HazardDataset,
    JRCFloodPolicy,
    curve_quantiles,
    curve_quantiles_at,
    return_periods_to_probabilities,
)

# Use the native interactive renderer; maintainers can opt into saved PNG previews.
STATIC_PREVIEW = os.environ.get("CRC_NOTEBOOK_STATIC_PREVIEW") == "1"
pio.renderers.default = (
    "colab" if IN_COLAB else "plotly_mimetype+png" if STATIC_PREVIEW else "plotly_mimetype"
)

if IN_COLAB:
    # Colab's default widget manager only trusts a fixed set of "core" ipywidgets
    # and silently fails to render anything else -- including anywidget-based
    # widgets, which is what go.FigureWidget is built on in Plotly 6.x. Without
    # this opt-in, every VBox containing a FigureWidget (the return-period
    # slider's chart, the four regional-analysis maps) never renders, even on
    # an otherwise successful, first-ever run.
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

H3_RESOLUTION = 7
RETURN_PERIOD_YEARS = 25  # curve-derived: EFAS has RP20 and RP30, but no RP25 raster
MIN_DEPTH_M = 0.5
SOURCE_RETURN_PERIODS = (10, 20, 30, 40, 50, 75, 100, 200, 500)

# One tuned connection config, reused by the indexer and our own joins.
duck_config = DuckDBConnection.for_analytics(DATA_DIR)
con = duck_config.connect()
indexer = H3Indexer(con)


## 1. Hazard: EFAS area → cached source crops → canonical curves

The [Joint Research Centre](https://joint-research-centre.ec.europa.eu)'s **River flood hazard maps for Europe and the Mediterranean** (CEMS-EFAS 3.1.1) publish nine continental return-period rasters. The plan below knows that current layout, resolves `latest` only when executed, pins the resolved release in its cache manifest, and stores AOI crops rather than complete continental TIFFs.

Acquisition periods and evaluation periods stay distinct: `.source_periods("all")` fits against all nine published knots, while `RETURN_PERIOD_YEARS` controls only the depth read from the canonical curve. RP25 demonstrates an estimate between the RP20 and RP30 source locations; slider requests above RP500 demonstrate extrapolation beyond fitted support. `explain()` inspects the plan without downloading rasters.

In [3]:
AOI_BOUNDS = (7.0, 49.7, 8.5, 50.9)  # Rhine corridor: Bonn/Cologne -> Koblenz -> Mainz/Wiesbaden

plan = (
    HazardDataset.efas(version="latest")
    .for_area(AOI_BOUNDS)
    .cache(DATA_DIR / "efas-source-cache", mode="reuse")
    .source_periods("all")
    .canonicalize(
        policy=JRCFloodPolicy.curated(h3_resolution=H3_RESOLUTION)
    )
)
print(plan.explain())

# The identity-addressed canonical cache is independent of evaluation RP.
# RP25 and every slider value therefore reuse the same fitted curves.
hazard_dataset = plan.ensure_materialized()
print(
    "Canonical curves:",
    "cache hit" if hazard_dataset.materialization is None else "built from source cache",
    f"({hazard_dataset.provider.source})",
)
metadata = hazard_dataset.metadata()
# Materialization already validates the canonical table. A direct Arrow read
# avoids starting a second validation process pool inside notebook kernels.
curves = pq.read_table(hazard_dataset.provider.source)
print("Canonical curve treatments:", curves["curve_kind"].value_counts().to_pylist())


Dataset: cems-efas-river-flood (continental)
Requested version: latest
Resolved version: 3.1.1
Area: 7.0,49.7,8.5,50.9
Source periods: 10,20,30,40,50,75,100,200,500
Cache: reuse (data/efas-source-cache)
Network access and fitting occur only at prefetch/materialize/write.
Canonical curves: cache hit (data/efas-source-cache/canonical/hazard-40c81b56b31630bcecca.parquet)
Canonical curve treatments: [{'values': 'fitted', 'counts': 154532}]


### Explore the canonical curve

The line is the fitted canonical Gumbel curve for one representative H3 cell.
Blue markers show that curve evaluated at the nine return periods for which JRC
publishes source rasters; they are **not raw raster observations**, which are
inputs to fitting and are not duplicated in the canonical row. Green selections
are curve-derived within RP10–RP500 support; red selections above RP500 are
extrapolations.

Drag the slider — it jumps only among useful source, interpolated, and
engineering return periods. Its handle carries no number (that's intentional,
working around a `SelectionSlider` crash on Google Colab's widget renderer) —
the status line and chart title below always show the selected RP instead. Both
update **in place**; they never redraw as a new chart. This same slider and
status line reappear in **Section 5** to drive the regional table and maps, so
nothing here needs re-running to use it later.

In [4]:
# Close any previous run's widgets first (best effort; a no-op on the very
# first run). This is just hygiene so re-running this cell during
# development doesn't leak stale comms -- it is NOT what makes re-running
# safe. display() below is unconditional and runs every time, because
# Jupyter/VS Code clears a cell's own previous output whenever it re-runs,
# so the *new* execution always needs its own fresh display() call; skipping
# it on the assumption "it's already displayed" just leaves the new (cleared)
# output area empty.
for _name in ("rp_slider", "selection_status", "curve_figure"):
    _old = globals().get(_name)
    if _old is not None:
        try:
            _old.close()
        except Exception:
            pass

# Pick the median RP100 curve from an evenly spaced sample of the region.
sample_rows = np.unique(np.linspace(0, len(curves) - 1, 401, dtype=int))
sample_curves = curves.take(sample_rows)
sample_probability = return_periods_to_probabilities(
    [100], tail=metadata.return_period_tail
)[0]
sample_depths = np.asarray(
    curve_quantiles_at(sample_curves, sample_probability, max_workers=1),
    dtype=float,
)
representative_index = int(np.nanargmin(np.abs(sample_depths - np.nanmedian(sample_depths))))
representative_curve = sample_curves.slice(representative_index, 1)
representative_cell = int(representative_curve["cell_index"][0].as_py())

curve_rps = np.linspace(10, 1000, 400)
curve_probabilities = return_periods_to_probabilities(
    curve_rps.tolist(), tail=metadata.return_period_tail
)
curve_depths = curve_quantiles(
    representative_curve, curve_probabilities, max_workers=1
)[0]
source_probabilities = return_periods_to_probabilities(
    SOURCE_RETURN_PERIODS, tail=metadata.return_period_tail
)
source_fitted_depths = curve_quantiles(
    representative_curve, source_probabilities, max_workers=1
)[0]

MEANINGFUL_RETURN_PERIODS = (10, 20, 25, 30, 40, 50, 75, 100, 200, 250, 500, 750, 1000)

def selected_return_period():
    return MEANINGFUL_RETURN_PERIODS[rp_slider.value]

# An IntSlider over the option *index*, not a SelectionSlider over the RP values
# themselves: SelectionSlider's `index` trait is a strict Int, and some frontends
# (Google Colab's custom widget renderer) sync it as a float with float-precision
# noise (e.g. 10.000000000000002), which raises an uncaught TraitError there.
# IntSlider.value is the lenient CInt trait, which tolerates that noise instead of
# crashing. readout is off because the raw index (0-12) isn't a year on its own;
# selected_return_period() and selection_status below show the actual RP instead.
rp_slider = widgets.IntSlider(
    min=0,
    max=len(MEANINGFUL_RETURN_PERIODS) - 1,
    value=MEANINGFUL_RETURN_PERIODS.index(RETURN_PERIOD_YEARS),
    description="Return period",
    continuous_update=False,
    readout=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="760px"),
)
selection_status = widgets.HTML()

# A FigureWidget is a live handle to one chart: the slider callback below mutates
# its traces in place, so the chart updates instead of being torn down and redrawn.
curve_figure = go.FigureWidget(
    data=[
        go.Scatter(
            x=curve_rps, y=curve_depths, mode="lines", name="Canonical fitted curve",
            line=dict(color="#555555", width=3),
        ),
        go.Scatter(
            x=list(SOURCE_RETURN_PERIODS), y=list(source_fitted_depths), mode="markers",
            name="Canonical fit at JRC source RPs",
            marker=dict(size=9, color="#1F77B4"),
        ),
        go.Scatter(
            x=[], y=[], mode="markers", name="Selected RP",
            marker=dict(size=14, line=dict(width=2, color="white")),
        ),
    ],
    layout=dict(
        xaxis=dict(title="Return period (years)", range=[10, 1000]),
        yaxis_title="Fitted flood depth (m)",
        margin=dict(l=60, r=20, t=60, b=50),
        height=430,
    ),
)

def render_curve_selection(change=None):
    rp = selected_return_period()
    selected_probability = return_periods_to_probabilities(
        [rp], tail=metadata.return_period_tail
    )[0]
    selected_depth = curve_quantiles(
        representative_curve, [selected_probability], max_workers=1
    )[0][0]
    if rp in SOURCE_RETURN_PERIODS:
        status = "canonical fit evaluated at a JRC source RP"
        color = "#1F77B4"
    elif rp <= metadata.return_period_support[1]:
        status = "canonical curve interpolation within source support"
        color = "#2CA02C"
    else:
        status = "canonical curve extrapolation beyond RP500"
        color = "#D62728"

    with curve_figure.batch_update():
        selected_trace = curve_figure.data[2]
        selected_trace.x = [rp]
        selected_trace.y = [selected_depth]
        selected_trace.marker.color = color
        selected_trace.name = f"Selected RP{rp}"
        curve_figure.layout.title = (
            f"Representative H3 cell {representative_cell}: RP{rp} = {selected_depth:.2f} m"
        )
    selection_status.value = (
        f"<b>RP{rp} selected</b> — {status} · AEP {100 / rp:.3g}% · fitted depth {selected_depth:.2f} m"
    )

rp_slider.observe(render_curve_selection, names="value")
display(widgets.VBox([rp_slider, selection_status, curve_figure]))
render_curve_selection()


## 2. Geography: geoBoundaries ADM1 → H3 polyfill

[geoBoundaries](https://www.geoboundaries.org) is a free, open (CC BY 4.0), globally consistent administrative boundary dataset — its ADM1/ADM2 attribute schema (`shapeID`/`shapeName`/`shapeGroup`) is already the convention crc-sdk's admin-lookup pipeline (`crc_sdk.geometry.admin`) is built around.

One call — `H3Indexer.build_h3_query_from_file` — reads Germany's simplified ADM1 GeoJSON straight to H3 cells at the same resolution as the hazard dataset.


In [5]:
ADM1_GEOJSON = DATA_DIR / "germany_adm1.geojson"
GEOBOUNDARIES_URL = (
    "https://github.com/wmgeolab/geoBoundaries/raw/9469f09/releaseData/"
    "gbOpen/DEU/ADM1/geoBoundaries-DEU-ADM1_simplified.geojson"
)

admin_cache_hit = ADM1_GEOJSON.exists()
if not admin_cache_hit:
    urllib.request.urlretrieve(GEOBOUNDARIES_URL, ADM1_GEOJSON)
print("ADM1 boundaries:", "cache hit" if admin_cache_hit else "downloaded")

adm1_h3_sql = indexer.build_h3_query_from_file(
    str(ADM1_GEOJSON),
    GeoFormat.GEOJSON,
    H3_RESOLUTION,
    PolyfillMode.OVERLAP,
    preserve_geom=False,
)
con.execute(f"CREATE OR REPLACE TEMP VIEW adm1_h3 AS {adm1_h3_sql}")
print(con.execute("SELECT COUNT(DISTINCT shapeName) FROM adm1_h3").fetchone())


ADM1 boundaries: cache hit
(16,)


## 3. Join hazard cells to provinces

`cell_index` (hazard) and `h3_index` (province polyfill) are both native unsigned H3 integers at the same resolution, so the spatial join is a plain `JOIN` — no resolution reconciliation needed. `hazard.depth_m` was already reconstructed at the requested return period in the previous cell (one curve evaluation per H3 cell, then reduced to each cell's worst case), so this join itself needs no further curve reconstruction.

## 4. Places: cached Overture AOI → impacted places

[Overture Maps](https://overturemaps.org) publishes a global places theme as
public GeoParquet. The first run resolves the current release and stores only
the filtered AOI points locally. Both the aggregation and PMTiles export reuse
that file, so warm runs perform no Overture request.

Each place is assigned to the same H3 grid as the hazard. Joining those counts
to the thresholded hazard cells shows how the selected event changes the set of
candidate places in scope. This is a coarse overlay, not exact source-pixel
exposure or proof of ownership.


In [6]:
OVERTURE_MIN_CONFIDENCE = 0.7
OVERTURE_RELEASE_FILE = DATA_DIR / "overture_release.txt"

if OVERTURE_RELEASE_FILE.exists():
    overture_release = OVERTURE_RELEASE_FILE.read_text().strip()
else:
    con.execute("SET s3_region = 'us-west-2'")
    overture_release = con.execute(
        "SELECT latest FROM read_json_auto('https://stac.overturemaps.org/catalog.json')"
    ).fetchone()[0]
    OVERTURE_RELEASE_FILE.write_text(overture_release)

cache_identity = json.dumps(
    {
        "release": overture_release,
        "bounds": AOI_BOUNDS,
        "min_confidence": OVERTURE_MIN_CONFIDENCE,
    },
    sort_keys=True,
)
places_cache_key = hashlib.sha256(cache_identity.encode()).hexdigest()[:12]
OVERTURE_POINTS_PATH = DATA_DIR / f"overture_places_{places_cache_key}.parquet"
places_cache_hit = OVERTURE_POINTS_PATH.exists()

if not places_cache_hit:
    con.execute("SET s3_region = 'us-west-2'")
    remote_places = (
        f"s3://overturemaps-us-west-2/release/{overture_release}"
        "/theme=places/type=place/*"
    )
    con.execute(
        f"""
        COPY (
            SELECT names.primary AS name, confidence,
                   ST_Point(bbox.xmin, bbox.ymin) AS geometry
            FROM read_parquet('{remote_places}', filename=true, hive_partitioning=1)
            WHERE bbox.xmin BETWEEN {AOI_BOUNDS[0]} AND {AOI_BOUNDS[2]}
              AND bbox.ymin BETWEEN {AOI_BOUNDS[1]} AND {AOI_BOUNDS[3]}
              AND confidence > {OVERTURE_MIN_CONFIDENCE}
        ) TO '{OVERTURE_POINTS_PATH}' (FORMAT PARQUET, COMPRESSION ZSTD)
        """
    )
print(
    f"Overture {overture_release} AOI points:",
    "cache hit" if places_cache_hit else "downloaded",
    f"({OVERTURE_POINTS_PATH})",
)

places_points_sql = f"""
    (SELECT name, confidence, geometry FROM read_parquet('{OVERTURE_POINTS_PATH}'))
"""
places_h3_sql = indexer.build_h3_query(
    places_points_sql,
    H3_RESOLUTION,
    PolyfillMode.CENTROID,
    geom_col="geometry",
    h3_col="h3_index",
    preserve_geom=False,
)


Overture 2026-08-19.0 AOI points: cache hit (data/overture_places_370fd8f2498b.parquet)


Four views over the same thresholded `risk`/`province_risk` frames: native-resolution hex depths, where Overture places concentrate inside those flood-exposed cells, the province roll-up, and a ranked comparison. Depth uses one sequential warm scale (YlOrRd) and places its own cool scale (Teal) throughout — each a magnitude, never a diverging quantity — and a dark basemap keeps low values legible instead of washing out against a light one.

JRC's own documentation flags that depths above ~10m near small channels can be modeling artifacts (DEM sinks or tile-boundary effects) rather than real hazard — worth keeping in mind for the deepest cells below.

**This section is self-contained.** The return-period slider and status line from Section 1 are repeated immediately below them — they're the same widgets, not copies, so dragging either instance moves both. Click **Run regional analysis** to re-join hazard, places, and province data for the selected RP; the table and all four charts below update **in place** — pan/zoom on the maps and any dataframe scroll position persist across runs, since only the underlying data changes, not the widgets themselves.

### PMTiles artifact

The same three geometry layers this notebook visualizes export as one [PMTiles](https://protomaps.com/docs/pmtiles) archive via `crc_sdk.geometry.PMTilesBuild`: one streaming tiling pass, not three separate archives. Its properties are deliberately presentation-ready and small: `hex_depth` carries only `depth_m` and `place_count`; `places` carries only `confidence`; and `provinces` carries its label plus the four regional metrics. IDs, source plumbing, and other non-visual fields stay out of the tiles. Hex boundaries come from DuckDB's vectorized `h3` extension; places retain the real point geometry from the cached AOI file.

**[⬇ Download risk_by_province.pmtiles](https://github.com/RiskThinking/crc-docs/blob/main/notebooks/artifacts/risk_by_province.pmtiles)** — small enough to commit directly (no Git LFS), so this link works straight from the repo without re-running the notebook. Drag it onto **[pmtiles.io](https://pmtiles.io)** (no server needed) to view it interactively. GitHub's notebook preview can't execute the JS a PMTiles viewer (MapLibre GL JS) needs, so this stays a downloadable artifact rather than an inline map.


In [7]:
# Close any previous run's widgets first (best effort; a no-op on the very
# first run) -- hygiene against leaking stale comms across reruns during
# development, not what makes re-running safe. rp_slider/selection_status
# are deliberately excluded: those are owned and recreated by Section 1, and
# only shared by reference here.
for _name in (
    "apply_button", "log_output", "table_output", "footer_output",
    "hex_figure", "places_figure", "province_figure", "bar_figure",
):
    _old = globals().get(_name)
    if _old is not None:
        try:
            _old.close()
        except Exception:
            pass

MAP_STYLE = "carto-darkmatter"  # dark basemap: low depths stay legible, unlike on a light one
MAP_VIEW = dict(map_zoom=7.3, map_center={"lat": 50.3, "lon": 7.75})
EMPTY_GEOJSON = {"type": "FeatureCollection", "features": []}

apply_button = widgets.Button(
    description="Run regional analysis",
    button_style="primary",
    icon="refresh",
    layout=widgets.Layout(width="240px"),
)
log_output = widgets.Output(
    layout=widgets.Layout(border="1px solid #ddd", padding="4px 8px", max_height="140px", overflow_y="auto")
)
table_output = widgets.Output()
footer_output = widgets.Output()

hex_figure = go.FigureWidget(
    data=[go.Choroplethmap(
        geojson=EMPTY_GEOJSON, locations=[], z=[],
        marker_line_width=0.5, marker_line_color="#999999", marker_opacity=0.95,
        colorscale="YlOrRd",
    )],
    layout=dict(map_style=MAP_STYLE, **MAP_VIEW, margin=dict(l=0, r=0, t=40, b=0), height=430),
)

places_figure = go.FigureWidget(
    data=[go.Scattermap(
        lat=[], lon=[], mode="markers",
        marker=dict(colorscale="Teal", showscale=True, colorbar_title="Places<br>(capped p90)"),
    )],
    layout=dict(
        map_style=MAP_STYLE, **MAP_VIEW, margin=dict(l=0, r=0, t=40, b=0), height=430,
        title="Overture places inside flood-exposed H3 cells",
    ),
)

with open(ADM1_GEOJSON) as handle:
    adm1_geojson = json.load(handle)

province_figure = go.FigureWidget(
    data=[go.Choroplethmap(
        geojson=adm1_geojson, locations=[], z=[],
        featureidkey="properties.shapeName", marker_line_width=1, marker_line_color="#999999",
        colorscale="YlOrRd", colorbar_title="Mean depth (m)",
    )],
    layout=dict(map_style=MAP_STYLE, **MAP_VIEW, margin=dict(l=0, r=0, t=40, b=0), height=430),
)

bar_figure = go.FigureWidget(
    data=[go.Bar(
        x=[], y=[], orientation="h", marker_color="#D94801",
        texttemplate="%{text} cells", textposition="outside", cliponaxis=False,
    )],
    layout=dict(xaxis_title="Mean depth (m)", margin=dict(l=140, r=60, t=40, b=40), height=380),
)


def run_regional_analysis(return_period_years=None):
    """Re-join hazard/places/provinces for one RP and update every widget above in place."""
    global RETURN_PERIOD_YEARS, hazard, risk, province_risk, risk_cells
    return_period_years = int(selected_return_period() if return_period_years is None else return_period_years)
    RETURN_PERIOD_YEARS = return_period_years

    log_output.clear_output(wait=True)
    def log(message):
        with log_output:
            print(message)

    if return_period_years > metadata.return_period_support[1]:
        log(f"RP{return_period_years} extrapolates beyond source support at RP500.")

    probability = return_periods_to_probabilities(
        [return_period_years], tail=metadata.return_period_tail
    )[0]
    evaluated_curves = pd.DataFrame(
        {
            "cell_index": curves["cell_index"].to_pylist(),
            "depth_m": curve_quantiles_at(curves, probability, max_workers=1),
        }
    )
    hazard = (
        evaluated_curves.groupby("cell_index", as_index=False)
        .max()
        .query("depth_m >= @MIN_DEPTH_M")
    )
    log(f"RP{return_period_years}: {len(hazard)} H3 cells at or above {MIN_DEPTH_M:.2f} m")

    con.register("hazard", hazard)
    risk = con.execute(
        """
        SELECT h.cell_index, a.shapeName AS province, h.depth_m
        FROM hazard h
        JOIN adm1_h3 a ON h.cell_index = a.h3_index
        """
    ).df()

    con.register("risk_cells", risk[["cell_index"]].drop_duplicates())
    impacted = con.execute(
        f"""
        WITH places_h3 AS ({places_h3_sql})
        SELECT p.h3_index AS cell_index, COUNT(*) AS place_count
        FROM places_h3 p
        JOIN risk_cells r ON p.h3_index = r.cell_index
        GROUP BY p.h3_index
        """
    ).arrow().read_all()

    impacted_by_cell = pd.DataFrame(impacted.to_pylist())
    candidate_places = int(impacted_by_cell["place_count"].sum())
    impacted_cell_count = len(impacted_by_cell)
    thresholded_cell_count = int(risk["cell_index"].nunique())
    risk = risk.merge(impacted_by_cell, on="cell_index", how="left")
    risk["place_count"] = risk["place_count"].fillna(0).astype(int)
    log(
        f"{candidate_places} Overture places across "
        f"{impacted_cell_count} of {thresholded_cell_count} unique thresholded cells"
    )

    province_risk = (
        risk.groupby("province")
        .agg(
            mean_depth_m=("depth_m", "mean"),
            max_depth_m=("depth_m", "max"),
            exposed_cells=("cell_index", "nunique"),
            impacted_places=("place_count", "sum"),
        )
        .reset_index()
        .sort_values("mean_depth_m", ascending=False)
    )
    table_output.clear_output(wait=True)
    with table_output:
        display(province_risk)

    # A handful of extreme (possibly artifact) cells would otherwise stretch the
    # scale so far that the other 90% of (still meaningful) depths all read as
    # the same pale color.
    depth_cap = risk["depth_m"].quantile(0.90)
    hex_geojson = {
        "type": "FeatureCollection",
        "features": [
            {"type": "Feature", "id": str(cell), "geometry": mapping(cell_polygon(cell))}
            for cell in risk["cell_index"]
        ],
    }
    with hex_figure.batch_update():
        trace = hex_figure.data[0]
        trace.geojson = hex_geojson
        trace.locations = risk["cell_index"].astype(str)
        trace.z = risk["depth_m"]
        trace.zmin = 0
        trace.zmax = depth_cap
        trace.customdata = risk[["province", "place_count"]]
        trace.hovertemplate = (
            "Province: %{customdata[0]}<br>Depth: %{z:.2f} m"
            "<br>Places: %{customdata[1]}<extra></extra>"
        )
        trace.colorbar.title.text = f"Depth (m)<br>{RETURN_PERIOD_YEARS}yr RP<br>(capped p90)"
        hex_figure.layout.title = (
            f"JRC riverine flood depth ≥ {MIN_DEPTH_M:.2f} m at the {RETURN_PERIOD_YEARS}-year return period "
            f"— Rhine corridor, H3 r{H3_RESOLUTION} cells"
        )

    # A cool hue (vs. the hazard map's warm YlOrRd) keeps the two magnitudes visually
    # distinct; a co-located overlay on the hex map would just bury the tiny hex
    # fills under bubbles at this zoom, so impacted places get their own map.
    impacted_cells = risk[risk["place_count"] > 0]
    centroids = [cell_polygon(cell).centroid for cell in impacted_cells["cell_index"]]

    # Place counts are just as right-skewed as depth (a few dense city cells vs.
    # many sparse ones), so cap the color scale the same way.
    place_cap = impacted_cells["place_count"].quantile(0.90)
    with places_figure.batch_update():
        trace = places_figure.data[0]
        trace.lat = [point.y for point in centroids]
        trace.lon = [point.x for point in centroids]
        trace.marker.size = 3 + np.log1p(impacted_cells["place_count"]) * 1.6
        trace.marker.color = impacted_cells["place_count"]
        trace.marker.cmin = 0
        trace.marker.cmax = place_cap
        trace.customdata = impacted_cells[["province", "place_count"]]
        trace.hovertemplate = (
            "Province: %{customdata[0]}<br>Places: %{customdata[1]}<extra></extra>"
        )

    with province_figure.batch_update():
        trace = province_figure.data[0]
        trace.locations = province_risk["province"]
        trace.z = province_risk["mean_depth_m"]
        trace.customdata = province_risk[["impacted_places"]]
        trace.hovertemplate = (
            "%{location}<br>Mean depth: %{z:.2f} m"
            "<br>Places: %{customdata[0]}<extra></extra>"
        )
        province_figure.layout.title = (
            f"Mean RP{RETURN_PERIOD_YEARS} depth by province (cells ≥ {MIN_DEPTH_M:.2f} m)"
        )

    ranked = province_risk.sort_values("mean_depth_m")
    with bar_figure.batch_update():
        trace = bar_figure.data[0]
        trace.x = ranked["mean_depth_m"]
        trace.y = ranked["province"]
        trace.text = ranked["exposed_cells"]
        trace.customdata = ranked[["impacted_places"]]
        trace.hovertemplate = (
            "%{y}<br>Mean depth: %{x:.2f} m"
            "<br>Places: %{customdata[0]}<extra></extra>"
        )
        bar_figure.layout.title = (
            f"Provinces ranked by mean RP{RETURN_PERIOD_YEARS} depth (cells ≥ {MIN_DEPTH_M:.2f} m)"
        )

    # Hex-depth layer: reuse the final risk rows, attach exact H3 boundary
    # geometry only at the very end (narrow-key-first idiom).
    con.register("risk_view", risk)
    hex_layer_path = DATA_DIR / "hex_layer.parquet"
    con.execute(
        f"""
        COPY (
            SELECT depth_m, place_count,
                   h3_cell_to_boundary_wkb(CAST(cell_index AS UBIGINT)) AS geometry
            FROM risk_view
        ) TO '{hex_layer_path}' (FORMAT PARQUET, COMPRESSION ZSTD)
        """
    )

    # Places layer: reuse the cached AOI points, keeping their real geometry and
    # restricting them to this scenario's thresholded hazard cells.
    places_layer_path = DATA_DIR / "places_layer.parquet"
    places_h3_geom_sql = indexer.build_h3_query(
        places_points_sql,
        H3_RESOLUTION,
        PolyfillMode.CENTROID,
        geom_col="geometry",
        h3_col="cell_index",
        preserve_geom=True,
    )
    con.execute(
        f"""
        COPY (
            SELECT p.confidence, p.geometry
            FROM ({places_h3_geom_sql}) p
            JOIN risk_cells r ON p.cell_index = r.cell_index
        ) TO '{places_layer_path}' (FORMAT PARQUET, COMPRESSION ZSTD)
        """
    )

    # Province-polygons layer: the same admin GeoJSON Section 2 already polyfilled,
    # read via the same format adapter H3Indexer uses internally.
    provinces_layer_path = DATA_DIR / "province_layer.parquet"
    admin_relation = FormatAdapter.build_read_relation(
        con, str(ADM1_GEOJSON), GeoFormat.GEOJSON, geometry_column="geometry",
        preserve_source_geom=False,
    )
    con.register("province_metrics", province_risk)
    con.execute(
        f"""
        COPY (
            SELECT a.shapeName AS province,
                   m.mean_depth_m, m.max_depth_m,
                   m.exposed_cells, m.impacted_places,
                   a.geometry
            FROM {admin_relation} a
            JOIN province_metrics m ON a.shapeName = m.province
        ) TO '{provinces_layer_path}' (FORMAT PARQUET, COMPRESSION ZSTD)
        """
    )

    # Layer intermediates stay under the gitignored DATA_DIR (scratch); the
    # final archive goes to ARTIFACTS_DIR instead -- unlike DATA_DIR, this one
    # is deliberately *not* gitignored, so the archive built below is small
    # enough to commit directly (no Git LFS) and downloadable straight from
    # the repo, not just reproducible by re-running the notebook.
    ARTIFACTS_DIR = Path("artifacts")
    ARTIFACTS_DIR.mkdir(exist_ok=True)
    PMTILES_PATH = ARTIFACTS_DIR / "risk_by_province.pmtiles"
    result = (
        PMTilesBuild(con=con)
        .layer(str(hex_layer_path), name="hex_depth", zooms=(0, 12), preset=AREAS)
        .add_layer(str(places_layer_path), name="places", zooms=(0, 14), preset=POINTS)
        .add_layer(str(provinces_layer_path), name="provinces", zooms=(0, 10), preset=POLYGONS)
        .write(str(PMTILES_PATH))
    )

    footer_output.clear_output(wait=True)
    with footer_output:
        print(
            f"wrote RP{return_period_years} scenario to {result.output} "
            f"({PMTILES_PATH.stat().st_size / 1024 / 1024:.1f} MiB, layers={result.layers})"
        )
        cached_inputs = {
            "canonical curves": Path(hazard_dataset.provider.source),
            "ADM1 boundaries": ADM1_GEOJSON,
            "Overture release": OVERTURE_RELEASE_FILE,
            "Overture AOI points": OVERTURE_POINTS_PATH,
        }
        missing_inputs = [name for name, cached_path in cached_inputs.items() if not cached_path.exists()]
        print("Reusable cache ready: " + ("yes" if not missing_inputs else f"no — missing {missing_inputs}"))
        display(FileLink(str(PMTILES_PATH), result_html_prefix="Download this run: "))

    return {
        "return_period_years": return_period_years,
        "thresholded_h3_cells": thresholded_cell_count,
        "candidate_places": candidate_places,
        "pmtiles": PMTILES_PATH,
    }


# Guards against duplicated output: `apply_button.disabled = True` only takes
# effect once the state change reaches the browser, so clicks already queued
# before that (e.g. impatient repeat-clicking during a slow run) would otherwise
# each re-enter this handler and re-run the full analysis. This flag makes any
# such extra, already-queued clicks a no-op instead of a duplicate run.
_analysis_running = False

def _apply_selected_return_period(_=None):
    global _analysis_running
    if _analysis_running:
        return
    _analysis_running = True
    apply_button.disabled = True
    apply_button.description = "Analyzing…"
    try:
        run_regional_analysis(selected_return_period())
    finally:
        _analysis_running = False
        apply_button.disabled = False
        apply_button.description = "Run regional analysis"


apply_button.on_click(_apply_selected_return_period)

# rp_slider/selection_status are the same widget objects from Section 1 (a shared
# model, not a copy): moving either view keeps both in sync, so this section never
# needs a scroll back up to see or change the current selection. display() below
# is unconditional -- see the matching comment in Section 1's cell for why.
display(widgets.VBox([
    rp_slider, selection_status, apply_button,
    log_output, table_output,
    hex_figure, places_figure, province_figure, bar_figure,
    footer_output,
]))

# Produce the default RP result once; every later slider move + button click
# reuses this same function and updates the widgets above in place.
_ = run_regional_analysis(RETURN_PERIOD_YEARS)


## Scaling this up

[`pipelines/flood_admin_pipeline.py`](https://github.com/RiskThinking/crc-docs/blob/main/pipelines/flood_admin_pipeline.py) is this notebook's headless twin — the same EFAS area selection → AOI cache → canonical fit → curve evaluation → admin join → places → PMTiles flow. Widen `--bounds` to scale the run; cached inputs are reused by default. Pass `--skip-pmtiles` on a machine without `tippecanoe`.